# v1 anomaly pipeline — train on normal, detect on unseen faults

Protocol:

1. **Train only on normal operation** (`faultNumber == 0`). Faulty runs never enter the fit.
2. Hold out a slice of that normal data as **validation** (false-alarm check / threshold).
3. Score **unseen fault runs** (`faultNumber > 0`) and ask: does the residual leave the normal envelope?

`te_process.csv` is the Rieth Tennessee Eastman set. The CSV column is `faultNumber` (not `fault_id`). `fault_status` is a **run-level** tag and is wrong as a sample label: test IDV(k) rows 1–160 are still healthy. Labels are rebuilt from `sample`.


In [ ]:
from __future__ import annotations

from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

CSV_PATH = Path("te_process.csv")
META_COLS = ["faultNumber", "simulationRun", "sample", "source", "fault_status"]

# How much of the 5.6 GB file to keep in RAM. Raise these to use more runs.
NORMAL_RUNS = 50          # train-source IDV(0) simulationRun 1..N
TEST_RUNS = 3             # test-source simulationRun 1..N
TEST_FAULTS = tuple(range(1, 21))  # IDV(1)–IDV(20)
KEEP_TEST_NORMAL = True   # extra healthy test trajectories for FAR
CHUNKSIZE = 250_000
NORMAL_FRACTION = 0.9     # train / (train+val) on normal rows

TRAIN_FAULT_SAMPLE = 21   # Rieth train files: disturbance at 1 h
TEST_FAULT_SAMPLE = 161   # Rieth test files: disturbance at 8 h
SAMPLE_MINUTES = 3

print("config")
print(f"  csv            {CSV_PATH.resolve()}")
print(f"  normal_runs    {NORMAL_RUNS}  (source=train, faultNumber=0)")
print(f"  test_runs      {TEST_RUNS}  (source=test, faults={TEST_FAULTS[0]}..{TEST_FAULTS[-1]})")
print(f"  normal_split   {NORMAL_FRACTION:.0%} train / {1 - NORMAL_FRACTION:.0%} val")


: 

## 1. Load trajectories

Stream the CSV and keep only the runs we asked for. Training data is `source="train"` and `faultNumber=0`. Evaluation data is `source="test"` so those simulations are disjoint from the fit.


In [ ]:
def sensor_columns(columns) -> list[str]:
    return [c for c in columns if c.startswith("xmeas_") or c.startswith("xmv_")]


def keep_mask(chunk: pd.DataFrame) -> pd.Series:
    source = chunk["source"]
    fault = chunk["faultNumber"].astype(int)
    run = chunk["simulationRun"].astype(int)
    train_ok = source.eq("train") & fault.eq(0) & run.between(1, NORMAL_RUNS)
    test_fault = source.eq("test") & fault.isin(TEST_FAULTS) & run.between(1, TEST_RUNS)
    test_normal = KEEP_TEST_NORMAL & source.eq("test") & fault.eq(0) & run.between(1, TEST_RUNS)
    return train_ok | test_fault | test_normal


def load_complete(counts: dict[tuple, int]) -> bool:
    train_ok = all(counts.get(("train", 0, r), 0) >= 500 for r in range(1, NORMAL_RUNS + 1))
    test_fault_ok = all(
        counts.get(("test", f, r), 0) >= 960
        for f in TEST_FAULTS
        for r in range(1, TEST_RUNS + 1)
    )
    test_normal_ok = (not KEEP_TEST_NORMAL) or all(
        counts.get(("test", 0, r), 0) >= 960 for r in range(1, TEST_RUNS + 1)
    )
    return train_ok and test_fault_ok and test_normal_ok


def load_pipeline_frame(path: Path = CSV_PATH) -> pd.DataFrame:
    """Keep a compact slice of the Rieth file: normal train runs + unseen test runs."""
    header = pd.read_csv(path, nrows=0)
    features = sensor_columns(header.columns)
    usecols = META_COLS + features
    parts: list[pd.DataFrame] = []
    counts: dict[tuple, int] = defaultdict(int)
    n_keep = 0

    for i, chunk in enumerate(pd.read_csv(path, usecols=usecols, chunksize=CHUNKSIZE), start=1):
        keep = keep_mask(chunk)
        if keep.any():
            taken = chunk.loc[keep]
            parts.append(taken)
            n_keep += len(taken)
            for key, n in taken.groupby(["source", "faultNumber", "simulationRun"], sort=False).size().items():
                counts[(key[0], int(key[1]), int(key[2]))] += int(n)
        if i == 1 or i % 8 == 0:
            print(f"  chunk {i:3d}  kept={n_keep:,}")
        if load_complete(counts):
            print(f"  chunk {i:3d}  kept={n_keep:,}  (all requested runs complete)")
            break

    if not parts:
        raise RuntimeError(f"no rows matched the load filters in {path}")

    out = pd.concat(parts, ignore_index=True)
    out["faultNumber"] = out["faultNumber"].astype(int)
    out["simulationRun"] = out["simulationRun"].astype(int)
    out["sample"] = out["sample"].astype(int)
    return out.sort_values(
        ["source", "faultNumber", "simulationRun", "sample"],
        kind="mergesort",
    ).reset_index(drop=True)


def fault_onset(source: str | pd.Series) -> np.ndarray | int:
    if isinstance(source, str):
        return TEST_FAULT_SAMPLE if source == "test" else TRAIN_FAULT_SAMPLE
    return np.where(np.asarray(source) == "test", TEST_FAULT_SAMPLE, TRAIN_FAULT_SAMPLE)


def add_true_label(df: pd.DataFrame) -> pd.DataFrame:
    """Sample-level label. Do not use CSV `fault_status`."""
    out = df.copy()
    onset = fault_onset(out["source"])
    out["y_true"] = ((out["faultNumber"] > 0) & (out["sample"] >= onset)).astype(int)
    return out


raw = add_true_label(load_pipeline_frame())
FEATURE_COLS = sensor_columns(raw.columns)

print("\nloaded")
print(raw.groupby(["source", "faultNumber"], sort=True).agg(
    n=("sample", "size"),
    runs=("simulationRun", "nunique"),
    smin=("sample", "min"),
    smax=("sample", "max"),
))
print(f"\nrows={len(raw):,}  sensors={len(FEATURE_COLS)}")
raw.head(3)


## 2. Train / val / test

`df_normal` is **only** `faultNumber == 0` from the train file. The 90/10 cut is the row split you sketched, then snapped to a `simulationRun` boundary so one trajectory is never split across train and val.

`test` is the unseen **test-file** fault runs. `test_normal` is the matching healthy test IDV(0) runs (false-alarm probe, not used in the fit).


In [ ]:
df_normal = (
    raw.loc[raw["source"].eq("train") & raw["faultNumber"].eq(0)]
    .sort_values(["simulationRun", "sample"], kind="mergesort")
    .reset_index(drop=True)
)

# Same idea as iloc[:90%] / iloc[90%:], but do not cut a run in half.
split = int(len(df_normal) * NORMAL_FRACTION)
run_at_split = int(df_normal.iloc[split]["simulationRun"])
split = int((df_normal["simulationRun"] == run_at_split).to_numpy().argmax())
# argmax finds the first True; that is the first row of the run that contains the 90% mark.
# that whole run (and everything after) goes to val.

train_normal = df_normal.iloc[:split].copy()
val_normal = df_normal.iloc[split:].copy()

test = raw.loc[raw["source"].eq("test") & raw["faultNumber"].gt(0)].copy()
test_normal = raw.loc[raw["source"].eq("test") & raw["faultNumber"].eq(0)].copy()


def _run_ids(df: pd.DataFrame) -> np.ndarray:
    return df["simulationRun"].drop_duplicates().to_numpy()


def _run_span(df: pd.DataFrame) -> str:
    runs = _run_ids(df)
    if len(runs) == 0:
        return "runs=[]"
    return f"runs={len(runs)} [{int(runs.min())}..{int(runs.max())}]"


print("split")
print(f"  df_normal     {len(df_normal):>8,} rows  {_run_span(df_normal)}")
print(f"  train_normal  {len(train_normal):>8,} rows  {_run_span(train_normal)}  faults={[int(x) for x in sorted(train_normal.faultNumber.unique())]}")
print(f"  val_normal    {len(val_normal):>8,} rows  {_run_span(val_normal)}  faults={[int(x) for x in sorted(val_normal.faultNumber.unique())]}")
print(f"  test          {len(test):>8,} rows  faults={[int(x) for x in sorted(test.faultNumber.unique())]}  {_run_span(test)}")
print(f"  test_normal   {len(test_normal):>8,} rows  {_run_span(test_normal)}")
print(f"  snapped split index={split:,}  ({split / max(len(df_normal), 1):.1%} of normal rows)")


In [ ]:
def overlap(a: pd.DataFrame, b: pd.DataFrame, cols=("source", "faultNumber", "simulationRun", "sample")) -> int:
    if a.empty or b.empty:
        return 0
    return int(pd.merge(a[list(cols)], b[list(cols)], how="inner").shape[0])


checks = {
    "train contains only faultNumber=0": train_normal["faultNumber"].eq(0).all(),
    "val contains only faultNumber=0": val_normal["faultNumber"].eq(0).all(),
    "train y_true is all 0": train_normal["y_true"].eq(0).all(),
    "val y_true is all 0": val_normal["y_true"].eq(0).all(),
    "no row overlap train/val": overlap(train_normal, val_normal) == 0,
    "no row overlap train/test": overlap(train_normal, test) == 0,
    "no shared simulationRun train/val": set(_run_ids(train_normal)).isdisjoint(_run_ids(val_normal)),
    "test has post-onset faulty samples": int(test["y_true"].sum()) > 0,
    "test has healthy prefix (y_true=0)": int((test["y_true"] == 0).sum()) > 0,
}

print("sanity")
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
if not all(checks.values()):
    raise AssertionError("split sanity checks failed")

print("\nlabel counts")
print("  train_normal  y_true", train_normal["y_true"].value_counts().to_dict())
print("  val_normal    y_true", val_normal["y_true"].value_counts().to_dict())
print("  test          y_true", test["y_true"].value_counts().sort_index().to_dict())
print("  test_normal   y_true", test_normal["y_true"].value_counts().to_dict())
print("\nCSV fault_status vs true label on test (first 8 h of IDV(k) are still healthy):")
print(pd.crosstab(test["fault_status"], test["y_true"], rownames=["fault_status"], colnames=["y_true"]))


## 3. Baseline detector (PCA-SPE)

Fit a PCA envelope on **train_normal only**. The SPE (squared prediction error) threshold is the 99th percentile of training residuals. Validation and healthy-test FAR should stay near 1%. Fault runs should leave that envelope after the disturbance onset.


In [ ]:
class PcaSpeDetector:
    """Unsupervised SPE detector. Fit exclusively on normal samples."""

    def __init__(self, variance=0.90, spe_quantile=0.99):
        self.variance = variance
        self.spe_quantile = spe_quantile
        self.mu_ = None
        self.sd_ = None
        self.P_ = None
        self.n_comp_ = None
        self.limit_ = None

    def fit(self, frame: pd.DataFrame, cols=None):
        cols = cols or FEATURE_COLS
        X = frame[cols].to_numpy(dtype=np.float64)
        self.mu_ = X.mean(axis=0)
        sd = X.std(axis=0)
        sd[sd == 0] = 1.0
        self.sd_ = sd
        Z = (X - self.mu_) / self.sd_
        _, S, Vt = np.linalg.svd(Z, full_matrices=False)
        explained = np.cumsum(S**2) / (S**2).sum()
        self.n_comp_ = int(np.searchsorted(explained, self.variance) + 1)
        self.P_ = Vt[: self.n_comp_].T
        spe = ((Z - Z @ self.P_ @ self.P_.T) ** 2).sum(axis=1)
        self.limit_ = float(np.quantile(spe, self.spe_quantile))
        return self

    def score(self, frame: pd.DataFrame, cols=None) -> pd.DataFrame:
        cols = cols or FEATURE_COLS
        Z = (frame[cols].to_numpy(dtype=np.float64) - self.mu_) / self.sd_
        spe = ((Z - Z @ self.P_ @ self.P_.T) ** 2).sum(axis=1)
        out = frame[META_COLS + ["y_true"]].copy()
        out["spe"] = spe
        out["alarm"] = spe > self.limit_
        return out


assert train_normal["faultNumber"].eq(0).all() and train_normal["y_true"].eq(0).all()

detector = PcaSpeDetector().fit(train_normal)
print(
    f"PCA-SPE  n_comp={detector.n_comp_}  "
    f"SPE {detector.spe_quantile:.0%} limit={detector.limit_:.2f}"
)

val_scores = detector.score(val_normal)
far_val = float(val_scores["alarm"].mean())
print(f"val FAR (should be ~{1 - detector.spe_quantile:.0%} if val matches train): {far_val:.3f}")

if not test_normal.empty:
    far_test_healthy = float(detector.score(test_normal)["alarm"].mean())
    print(f"test IDV(0) FAR: {far_test_healthy:.3f}")


In [ ]:
test_scores = detector.score(test)
pre = test_scores["y_true"].eq(0)
post = test_scores["y_true"].eq(1)

far_pre = float(test_scores.loc[pre, "alarm"].mean()) if pre.any() else np.nan
fdr_post = float(test_scores.loc[post, "alarm"].mean()) if post.any() else np.nan
print(f"unseen fault runs  FAR (healthy prefix): {far_pre:.3f}")
print(f"unseen fault runs  FDR (after onset):    {fdr_post:.3f}")


def first_alarm_delay(g: pd.DataFrame) -> float:
    onset = TEST_FAULT_SAMPLE
    hits = g.loc[g["sample"] >= onset, "alarm"]
    if not hits.any():
        return np.nan
    first = int(g.loc[g["sample"] >= onset].loc[hits.to_numpy(), "sample"].iloc[0])
    return (first - onset) * SAMPLE_MINUTES


by_fault = (
    test_scores.groupby("faultNumber", sort=True)
    .apply(
        lambda g: pd.Series(
            {
                "runs": g["simulationRun"].nunique(),
                "FAR_prefix": g.loc[g.y_true.eq(0), "alarm"].mean(),
                "FDR_post": g.loc[g.y_true.eq(1), "alarm"].mean(),
                "median_delay_min": g.groupby("simulationRun").apply(first_alarm_delay).median(),
            }
        ),
        include_groups=False,
    )
)
print("\ndetection by IDV (unseen test runs)")
display(by_fault.round(3))


In [ ]:
# One trajectory: first unseen IDV(1) run. SPE should stay under the limit for 8 h, then jump.
example = test_scores.loc[
    test_scores["faultNumber"].eq(1) & test_scores["simulationRun"].eq(int(_run_ids(test)[0]))
].sort_values("sample")
hours = (example["sample"] - 1) * SAMPLE_MINUTES / 60
onset_h = (TEST_FAULT_SAMPLE - 1) * SAMPLE_MINUTES / 60

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(hours, example["spe"], lw=1.2, label="SPE")
ax.axhline(detector.limit_, color="C1", ls="--", lw=1, label=f"limit ({detector.spe_quantile:.0%} train)")
ax.axvline(onset_h, color="C3", ls=":", lw=1.4, label="fault onset (8 h)")
ax.set_xlabel("hours from start of test run")
ax.set_ylabel("SPE")
ax.set_title(f"IDV(1) simulationRun={int(example['simulationRun'].iloc[0])} — never seen in training")
ax.legend(loc="upper left")
ax.set_xlim(0, hours.max())
fig.tight_layout()
plt.show()

first_hit = example.loc[example["sample"].ge(TEST_FAULT_SAMPLE) & example["alarm"], "sample"]
if first_hit.empty:
    print("no alarm after onset on this run")
else:
    delay = (int(first_hit.iloc[0]) - TEST_FAULT_SAMPLE) * SAMPLE_MINUTES
    print(f"first post-onset alarm at sample {int(first_hit.iloc[0])}  ({delay} min after onset)")
